# 🙂 Demo: eigenfaces, PCA con rostros

> Material de respaldo de la sesión de PCA: **no se trabaja en clase**. Ejecútalo de arriba a abajo y lee las observaciones.

**Labeled Faces in the Wild (LFW)** es una base pública de fotos de personajes públicos tomadas de noticias. Usamos las personas con al menos 60 fotos. Cada componente principal de un conjunto de rostros es, a su vez, una "cara": se le llama **eigenface**.

⚠️ **Descarga:** la primera vez `fetch_lfw_people` descarga unos **200 MB** (puede tardar varios minutos). Después queda guardado en `~/scikit_learn_data` y carga en segundos. Si la descarga falla por la red, vuelve a ejecutar la celda más tarde.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.datasets import fetch_lfw_people
from sklearn.decomposition import PCA

caras = fetch_lfw_people(min_faces_per_person=60, resize=0.4)
alto, ancho = caras.images.shape[1:]
print(f"Fotos: {caras.images.shape[0]:,} de {len(caras.target_names)} personas")
print(f"Cada foto: {alto} x {ancho} = {caras.data.shape[1]:,} píxeles")
print(f"Personas: {', '.join(caras.target_names)}")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.8))
for i, ax in enumerate(axes.flat):
    ax.imshow(caras.images[i], cmap='gray')
    ax.set_title(caras.target_names[caras.target[i]].split()[-1], fontsize=8)
    ax.axis('off')
plt.suptitle('Algunas fotos de LFW')
plt.show()

---
## 1️⃣ ¿Cuántos componentes necesitan los rostros?

Con tantas columnas usamos `svd_solver='randomized'`: calcula sólo los primeros componentes, mucho más rápido que la descomposición completa. Como en MNIST, los píxeles comparten unidad: no estandarizamos.

In [ ]:
pca_caras = PCA(n_components=250, svd_solver='randomized', random_state=42).fit(caras.data)
acumulada = np.cumsum(pca_caras.explained_variance_ratio_)

fig = px.line(x=np.arange(1, 251), y=acumulada,
              labels={'x': 'Número de componentes', 'y': 'Varianza acumulada'},
              title='LFW: varianza acumulada (primeros 250 componentes)')
fig.update_layout(width=800, height=420, yaxis_tickformat='.0%')
fig.show()
for k in (5, 15, 50, 100, 150, 250):
    print(f"{k:>3} componentes → {acumulada[k - 1]:.1%} de la varianza")

✅ **Qué observar:**
- Con **50** componentes ya tenemos el **84.4%** de la varianza; con **150**, el **94.3%**
- 150 componentes de **1,850** píxeles: una reducción de más de **12×**

---
## 2️⃣ La cara promedio y las eigenfaces

PCA centra los datos: primero calcula la **cara promedio** (`mean_`). Cada eigenface dice **en qué se desvía** una cara del promedio.

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(12, 5.4))
axes.flat[0].imshow(pca_caras.mean_.reshape(alto, ancho), cmap='gray')
axes.flat[0].set_title('Promedio', fontsize=9)
for i, ax in enumerate(axes.flat[1:]):
    ax.imshow(pca_caras.components_[i].reshape(alto, ancho), cmap='bone')
    ax.set_title(f'PC{i + 1}', fontsize=9)
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Cara promedio y las primeras 23 eigenfaces')
plt.show()

✅ **Qué observar:**
- Las primeras eigenfaces capturan variaciones **globales**: de qué lado viene la luz, si la cara está girada, el contorno del cabello
- Las siguientes capturan detalles más **finos**: ojos, nariz, boca, sombras locales
- Igual que en el vino, cada componente es una **mezcla** de todas las variables (aquí, de todos los píxeles)

---
## 3️⃣ Reconstrucción progresiva de un rostro

Proyectamos una foto a 250 componentes y la reconstruimos usando sólo los primeros $k$.

In [ ]:
foto = caras.data[[0]]
scores_foto = pca_caras.transform(foto)
niveles = [5, 15, 50, 100, 150, 250]

fig, axes = plt.subplots(1, len(niveles) + 1, figsize=(14, 3.2))
axes[0].imshow(foto.reshape(alto, ancho), cmap='gray')
axes[0].set_title(f'Original\n{caras.data.shape[1]:,} px', fontsize=9)
for ax, k in zip(axes[1:], niveles):
    rec = scores_foto[:, :k] @ pca_caras.components_[:k] + pca_caras.mean_
    ax.imshow(rec.reshape(alto, ancho), cmap='gray')
    ax.set_title(f'{k} comp.\n({acumulada[k - 1]:.0%} var.)', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.suptitle('Reconstrucción con distintos números de componentes')
plt.show()

✅ **Qué observar:**
- Con **5** componentes sólo hay una silueta con la iluminación correcta
- Con **50** (84%) la persona ya es reconocible; con **150** (94%) la calidad es muy aceptable
- 💡 Las eigenfaces fueron una de las primeras técnicas de reconocimiento facial (Turk y Pentland, 1991): comparar personas con 150 números en lugar de 1,850 píxeles

## 📝 En resumen
1. Con imágenes, cada componente principal se puede **ver**: es una plantilla del mismo tamaño que la foto
2. Los primeros componentes capturan lo global (luz, pose) y los últimos, el detalle fino
3. `svd_solver='randomized'` acelera PCA cuando sólo necesitas los primeros componentes de datos muy anchos